In [2]:
# 🧹 Task 1 – Data Processing & Cleaning Notebook
# This notebook performs NLP-oriented data preprocessing on the complaints dataset
# It prepares the dataset for embedding, vector database loading, and RAG tasks.

# 1️⃣ Import Libraries
import pandas as pd
import numpy as np
import re
import string
from pathlib import Path

# 2️⃣ Load Dataset
RAW_PATH = '../data/raw/complaints.csv'
df = pd.read_csv(RAW_PATH)
print("Dataset Shape:", df.shape)
print(df.head())

# 3️⃣ Basic Cleaning
df.columns = df.columns.str.strip()

# Remove empty narratives
df = df[df['Consumer complaint narrative'].notna()]
df = df[df['Consumer complaint narrative'].str.strip() != '']
print("After removing empty narratives:", df.shape)

# 4️⃣ Normalize Product Categories
mapping_rules = {
    'Personal loan': [
        'Payday loan, title loan, personal loan, or advance loan',
        'Student loan',
        'Payday loan, title loan, or personal loan',
        'Consumer Loan'
    ],
    'Money transfers': [
        'Money transfer, virtual currency, or money service',
        'Money transfers'
    ],
    'Savings account': [
        'Checking or savings account'
    ],
    'Credit card': [
        'Credit card or prepaid card',
        'Credit card'
    ]
}

def map_product(value):
    if pd.isna(value):
        return None
    value = value.strip()
    for category, values in mapping_rules.items():
        if value in values:
            return category
    return None

df['Product_Cleaned'] = df['Product'].apply(map_product)
df = df[df['Product_Cleaned'].notna()]
print("Filtered to target products:", df.shape)
print(df['Product_Cleaned'].value_counts())

# 5️⃣ Text Cleaning Function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)              # remove HTML
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    text = re.sub(r'\s+', ' ', text)                # remove extra spaces
    return text.strip()

df['Complaint_Text'] = df['Consumer complaint narrative'].astype(str).apply(clean_text)

# 6️⃣ Remove Extremely Short Complaints
df['text_length'] = df['Complaint_Text'].str.len()
df = df[df['text_length'] > 50]
print("After removing very short complaints:", df.shape)
print(df['text_length'].describe())

# 7️⃣ Select Final Columns
final_df = df[[
    'Complaint ID',
    'Date received',
    'Product_Cleaned',
    'Company',
    'State',
    'Complaint_Text'
]]

print("Final processed dataset preview:")
print(final_df.head())



# ✅ Summary
print("Processed dataset ready for Task 2 - RAG Pipeline!")


C:\Users\derej\AppData\Local\Temp\ipykernel_20612\1321660078.py:14: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(RAW_PATH)


Dataset Shape: (9609797, 18)
  Date received                                            Product  \
0    2025-06-20  Credit reporting or other personal consumer re...   
1    2025-06-20                                    Debt collection   
2    2025-06-20  Credit reporting or other personal consumer re...   
3    2025-06-20  Credit reporting or other personal consumer re...   
4    2025-06-20  Credit reporting or other personal consumer re...   

               Sub-product                                 Issue  \
0         Credit reporting  Incorrect information on your report   
1  Telecommunications debt     Attempts to collect debt not owed   
2         Credit reporting           Improper use of your report   
3         Credit reporting           Improper use of your report   
4         Credit reporting  Incorrect information on your report   

                                       Sub-issue Consumer complaint narrative  \
0            Information belongs to someone else            